> Experiments with [model.signal](../trapallada/model/signal.py)  module

In [1]:
import pathlib
import pprint

import yaml
# import numpy as np
import pandas as pd
# from IPython.display import display

# import jax
# import jax.numpy as jnp
import jax.random as jr

import trapallada.data
import trapallada.model.signal
import trapallada.model.image
import trapallada.util
import trapallada.core

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Directory containing the data

In [3]:
input_directory = pathlib.Path.cwd().parent / 'data'
assert input_directory.exists()
input_directory

PosixPath('/home/manu/Sync/git/trapallada/data')

Data is split in two different files:

- data itself, in the form of an `Array`

- metadata in a *Pandas* `DataFrame`

In [4]:
base_filename = '4_classes_modulated'

# signals_file = input_directory / f'{base_filename}_signals.pt'
signals_file = input_directory / '20_600_signals.pt'
assert signals_file.exists()

# metadata_file = input_directory / f'{base_filename}_labels.parquet'
metadata_file = input_directory / f'20_600_metadata.parquet'
assert metadata_file.exists

In [5]:
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, normalize=True)
# ds = trapallada.data.SignalsDataset(signals_file, metadata_file, normalize=False)
len(ds)

trapallada.data - WARNING - dismissed 349 constant signals


35317

In [6]:
# parameters_file = pathlib.Path.cwd().parent / 'parameters_vae_on_signals.yaml'
parameters_file = pathlib.Path.cwd().parent / 'experiments' / 'new' / 'real_z_50_300_epochs.yaml'
assert parameters_file.exists()

In [7]:
with parameters_file.open() as f:
    parameters = yaml.safe_load(f)

In [8]:
# pprint.pprint(parameters)
%pycat {parameters_file}

PRNG seeds:
  model initialization:
    decoder: 43
    encoder: 42
  sampling:
    training set: 14
    validation set: 15
  training dataloader: 43
  training-validation splitting: 42
encoder:
  input_size:
  - 1
  - 600
loss:
  x_sd: 0.25
model:
  channels:
  - 12
  - 12
  - 12
  dim_z: 20 
  kernel_sizes:
  - 128
  - 64
  - 32
  stride: 2
training:
  batch_size: 32
  dl_rng_key: 43
  learning_rate: 3.0e-05
  n_epochs: 300
  train_test_split:
    test_size: 0.2


In [9]:
train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(
    ds,
    parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    parameters['training']['batch_size'], parameters['PRNG seeds']['training dataloader'])


In [10]:
batch = next(iter(train_dl))
len(batch)

3

In [11]:
batch[0]

tensor([21187,  3059,  8863,  6706, 23290, 31705,  9756, 19072, 33351, 34676,
         8559, 29299, 20400, 30410, 13248, 30155, 13064, 34028, 13947, 11447,
        19923, 24394, 17348,  3494, 17953, 12217, 29802, 12988,  3795,  9027,
        17347, 30234])

In [12]:
signals_len = ds[0][1].shape[-1]
signals_len

600

In [13]:
tuple(ds[0][1].shape)

(1, 600)

Input size must be adjusted

In [14]:
parameters['encoder']['input_size'] = list(ds[0][1].shape)

In [15]:
x = batch[1][0].numpy()
encoder = trapallada.model.signal.Encoder(
    **(parameters['model'] | parameters['encoder']),
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['encoder']))
encoder.debug_call(x)
print(encoder)

after equinox.nn._conv: (12, 237)
after jax._src.nn.functions: (12, 237)
after equinox.nn._normalisation: (12, 237)
after equinox.nn._conv: (12, 87)
after jax._src.nn.functions: (12, 87)
after equinox.nn._normalisation: (12, 87)
after equinox.nn._conv: (12, 28)
after jax._src.nn.functions: (12, 28)
after equinox.nn._normalisation: (12, 28)
after jax._src.numpy.lax_numpy: (336,)
after equinox.nn._linear: (40,)
after trapallada.model.signal: (40,)
Encoder(
  layers=[
    Conv1d(
      num_spatial_dims=1,
      weight=f32[12,1,128],
      bias=f32[12,1],
      in_channels=1,
      out_channels=12,
      kernel_size=(128,),
      stride=(2,),
      padding=((0, 0),),
      dilation=(1,),
      groups=1,
      use_bias=True,
      padding_mode='ZEROS'
    ),
    <wrapped function relu>,
    LayerNorm(
      shape=(12, 237),
      eps=1e-05,
      use_weight=True,
      use_bias=True,
      weight=f32[12,237],
      bias=f32[12,237]
    ),
    Conv1d(
      num_spatial_dims=1,
      weight=f

In [16]:
encoder.layers_spatial_sizes

[600, 237, 87, 28, 336]

In [17]:
z = encoder(x)
z.shape

(40,)

In [18]:
z_mean = jr.normal(jr.PRNGKey(42), (parameters['model']['dim_z'],))
decoder = trapallada.model.signal.Decoder(
    **(parameters['model'] | {'layers_spatial_sizes': encoder.layers_spatial_sizes}),
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['decoder']))
decoder.debug_call(z_mean)

after equinox.nn._linear: (336,)
after jax._src.nn.functions: (336,)
after trapallada.model.signal: (12, 28)
after equinox.nn._conv: (12, 87)
after jax._src.nn.functions: (12, 87)
after equinox.nn._normalisation: (12, 87)
after equinox.nn._conv: (12, 237)
after jax._src.nn.functions: (12, 237)
after equinox.nn._normalisation: (12, 237)
after equinox.nn._conv: (1, 600)


In [19]:
vae = trapallada.model.signal.VAE(encoder, decoder)
x_pred, z_mean, z_sd = vae(x, jr.PRNGKey(42))
x_pred.shape, z_mean.shape, z_sd.shape

((1, 600), (20,), (20,))

In [20]:
x_pred

Array([[-0.03261267, -0.04187645, -0.03879522, -0.0819178 , -0.09101183,
        -0.05603087, -0.11331206, -0.03833373, -0.10700347, -0.01627006,
        -0.10159094,  0.00858253, -0.0652959 ,  0.01794218, -0.10369501,
         0.00836701, -0.12374   ,  0.06158667, -0.19868484,  0.06063124,
        -0.13046633,  0.07232925, -0.11202601,  0.04478004, -0.13386409,
         0.16362783, -0.16782473,  0.02976014, -0.13321067,  0.00198699,
        -0.22571872,  0.02445945, -0.09807753,  0.1358798 , -0.12931868,
         0.04637624, -0.2543214 , -0.01845551, -0.1739324 , -0.06158118,
        -0.16909435,  0.03564446, -0.25963515, -0.17518067, -0.06247631,
         0.03373454, -0.20992103,  0.01338315, -0.24929422,  0.0380781 ,
        -0.2945878 ,  0.06326525, -0.32414645,  0.01384726, -0.43779826,
        -0.01694426, -0.2558065 , -0.03249098, -0.14398165,  0.00887807,
        -0.16953497, -0.03547326, -0.08194245, -0.07705244, -0.08092181,
         0.19680308, -0.10621023, -0.0188027 , -0.1

In [23]:
# model, results_df = trapallada.core.fit_vae_on_signals(train_dl, validation_dl, parameters)